In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import glob

project_root =Path("C:/Users/axere/OneDrive/Desktop/CycleDA/cycle_data_13to22")

raw_folder = project_root / "Cycle_Data" / "raw_data"
cleaned_folder = project_root / "Cycle_Data" / "Cleaned_data"
meta_folder = project_root / "Cycle_Data" / "meta_data"

cleaned_folder.mkdir(exist_ok=True)
meta_folder.mkdir(exist_ok=True)


def extract_stations_from_trip_file(df):
    
    start_stations = df[[
        "from_station_id",
        "from_station_name",
        "start_lat",
        "start_long"
    ]].rename(columns={
        "from_station_id": "ID",
        "from_station_name": "station_name",
        "start_lat": "station_lat",
        "start_long": "station_long"
    })

    end_stations = df[[
        "to_station_id",
        "to_station_name",
        "end_lat",
        "end_long"
    ]].rename(columns={
        "to_station_id": "ID",
        "to_station_name": "station_name",
        "end_lat": "station_lat",
        "end_long": "station_long"
    })

    stations = pd.concat([start_stations, end_stations], ignore_index=True)

    return stations

new_station_records = []

def clean_trip_data(df, year):

        if len(df.columns) not in [12,13]:
            print("Skipping unexpected file schema")
            return None

        if len(df.columns) == 12:
            schema = "2013-2019"
        elif len(df.columns) == 13:
            schema = "2020-2022"

        if schema == "2013-2019":

            # Renaming columns for better format
            col_name = ["trip_id",	"start_time",	"stop_time",	"bike_id",	"trip_duration",	"from_station_id",	"from_station_name",	"to_station_id",	"to_station_name",	"usertype",	"gender", "birth_year"]

            df.columns = col_name
                            
            # Checking for null values in trip_id
            null_sum = df["trip_id"].isnull().sum()
            if (null_sum >=1):
                df= df.dropna(subset=["trip_id"])

            # Checking for duplicates in trip_id
            duplicate_sum = df["trip_id"].duplicated().sum()
            if (duplicate_sum >=1):
                df= df.drop_duplicates(subset=["trip_id"])

            # If the trip id repeats in other years  
            df["unique_trip_id"] = str(year) +'_'+ df["trip_id"].astype(str)

            # Strandardizing the time and date
            df["start_time"] = pd.to_datetime(df["start_time"],format = "mixed", dayfirst = True, errors="coerce")
            df["stop_time"] = pd.to_datetime(df["stop_time"],format = "mixed", dayfirst = True, errors="coerce")
            df = df.dropna(subset=["start_time", "stop_time"])


            # Creating a min column for easier calcualtions
            df["trip_duration"] = pd.to_numeric(df["trip_duration"], errors="coerce")
            df = df.dropna(subset=["trip_duration"])
            df["trip_duration_min"] = (df["trip_duration"]/60).round(2)
                        
            # Creating different columns for year, month and day for future analysis 
            df["trip_year"] = year
            df["trip_month"] = df["start_time"].dt.strftime('%b')
            df["trip_day"] = df["start_time"].dt.strftime('%a')

            # Checking for invalid time such as when start time is ahead of stop time
            invalid_time = df[df["start_time"]>df["stop_time"]].index
            df = df.drop(invalid_time)

            # Checking if the trip_duration is within 60 sec of start stop time (To check for inconsistant trip_duration)
            df["duration_check"] = (df["stop_time"] - df["start_time"]).dt.total_seconds()
            high_diff = df[(df["duration_check"] - df["trip_duration"])>60].index
            df = df.drop(high_diff)

            df = df.drop("duration_check",axis=1)

            # Creating an age column for checking valid age
            df["birth_year"] = pd.to_numeric(df["birth_year"], errors="coerce")
            df["age"] = ((df["start_time"].dt.year) - df["birth_year"])

            # Setting invalid age as null
            df.loc[df["age"]>90, "birth_year"] = np.nan
            df.loc[df["age"]>90, "age"] = np.nan

            # Checking for invalid trip
            unrealistic_trips = df[(df["trip_duration_min"] > 720 )].index
            df = df.drop(unrealistic_trips)

            df= df.sort_values(by="start_time")
                        
        if schema == "2020-2022":
            col_name = ["trip_id", "ride_type", "start_time", "stop_time", "from_station_name", "from_station_id","to_station_name", "to_station_id", "start_lat", "start_long", "end_lat", "end_long", "usertype"]

            df.columns = col_name 

            stations_df = extract_stations_from_trip_file(df)
            new_station_records.append(stations_df)


            # Checking for null values in trip_id
            null_sum = df["trip_id"].isnull().sum()
            if (null_sum >=1):
                df= df.dropna(subset=["trip_id"])

            # Checking for duplicates in trip_id    
            duplicate_sum = df["trip_id"].duplicated().sum()
            if (duplicate_sum >=1):
                df= df.drop_duplicates(subset=["trip_id"])

            # If the trip id repeats in other years 

            df["unique_trip_id"] = str(year) +'_'+ df["trip_id"].astype(str)

            # Strandardizing the time and date
            df["start_time"] = pd.to_datetime(df["start_time"],format = "mixed", dayfirst = True, errors="coerce")
            df["stop_time"] = pd.to_datetime(df["stop_time"],format = "mixed", dayfirst = True, errors="coerce")
            df = df.dropna(subset=["start_time", "stop_time"])

            # Creating different columns for year, month and day for future analysis
            df["trip_year"] = year
            df["trip_month"] = df["start_time"].dt.strftime('%b')
            df["trip_day"] = df["start_time"].dt.strftime('%a')

            df["start_lat"] = pd.to_numeric(df["start_lat"], errors="coerce")
            df["start_long"] = pd.to_numeric(df["start_long"], errors="coerce")
            df["end_lat"] = pd.to_numeric(df["end_lat"], errors="coerce")
            df["end_long"] = pd.to_numeric(df["end_long"], errors="coerce")
            df = df.dropna(subset=["start_lat","start_long","end_lat","end_long"])
            df["trip_duration_min"] = ((df["stop_time"]- df["start_time"]).dt.total_seconds()/60).round(2)


            # Checking for invalid time such as when start time is ahead of stop time
            invalid_time = df[df["start_time"]>df["stop_time"]].index
            df = df.drop(invalid_time)

            # Checking if the trip_duration is within 60 sec of start stop time (To check for inconsistant trip_duration)
            df["duration_check"] = (df["stop_time"] - df["start_time"]).dt.total_seconds()
            high_diff = df[((df["duration_check"]/60) - df["trip_duration_min"])>1].index
            df = df.drop(high_diff)

            df = df.drop("duration_check",axis=1)

            # Checking for invalid trip
            unrealistic_trips = df[(df["trip_duration_min"] > 720 )].index
            df = df.drop(unrealistic_trips)

            df= df.sort_values(by="start_time")

            df = df.drop(columns=[
                "from_station_name",
                "to_station_name",
                "start_lat",
                "start_long",
                "end_lat",
                "end_long"
            ])

        return df

for file in raw_folder.glob("*.csv"):
    print("Processing:", file.name)
    df = pd.read_csv(file)
    if file.name.startswith("s"):
        print(f"Skipping metadata file: {file.name}")
        continue
    year = "20"+file.stem[:2]

    cleaned_df = clean_trip_data(df,year)
    if cleaned_df is None:
        continue


output_file = cleaned_folder / f"{year}_{file.stem}_cleaned.csv"
cleaned_df.to_csv(output_file, index = False)
print(f"Saved: {output_file}")

extracted_stations = pd.concat(new_station_records, ignore_index=True)
extracted_stations["ID"] = extracted_stations["ID"].astype(str).str.strip()

# for meta data from s13-s17b
import pandas as pd
import glob

meta_files = glob.glob("C:/Users/axere/OneDrive/Desktop/CycleDA/cycle_data_13to22/Cycle_Data/raw_data/s*.csv")

dfs=[]
for files in meta_files:

    df = pd.read_csv(files)

    df.columns = (
        df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
            .str.replace("-", "_"))
    
    rename_col = {"id":"ID",
                "name": "station_name",
                "city": "city_loc",
                "latitude": "station_lat",
                "longitude": "station_long",
                "dpcapacity": "dp_capacity",
                "landmark": "landmark",

                "online_date":"station_opening_date",
                "onlinedate": "station_opening_date",
                "datecreated": "station_opening_date"}

    df = df.rename(columns=rename_col)
    df["ID"] = df["ID"].astype(str).str.strip()
    col_name = ["ID","station_name","city_loc","station_lat","station_long","dp_capacity","landmark","station_opening_date"]

    for col in col_name:
        if col not in df.columns:
            df[col] = None

    dfs.append(df)

all_stations = pd.concat(dfs,ignore_index=True)
all_stations = pd.concat([all_stations, extracted_stations], ignore_index=True)
if "_7" in all_stations:
    all_stations= all_stations.drop("_7", axis = 1)
if "unnamed:_7" in all_stations:
    all_stations= all_stations.drop("unnamed:_7", axis = 1)

all_stations["ID"] = all_stations["ID"].astype(str).str.strip()

all_stations["station_opening_date"] = pd.to_datetime(
    all_stations["station_opening_date"],
    errors="coerce"
)

all_stations["info_score"] = all_stations.notna().sum(axis=1)

stations_master = (
    all_stations
    .sort_values("info_score", ascending=False)
    .drop_duplicates(subset="ID", keep="first")
    .drop(columns="info_score")
    .reset_index(drop=True)
)
stations_master = stations_master.reset_index(drop=True)

stations_master.insert(0, "station_pk", stations_master.index + 1)

stations_master = stations_master.rename(columns={"ID": "station_id"})
meta_output = meta_folder / f"merged_meta_data.csv"
stations_master.to_csv(meta_output , index = False)
print(f"saved :{meta_output}")